# Module 1 — Python & AI Development Environment Setup
## Google Colab Edition | Demonstrations 1, 2 and 3

**Class flow:** Set up a Python AI project → securely configure an LLM API → run independent LLM calls concurrently.

**Before you begin**
1. Open this notebook in Google Colab.
2. Use **Runtime → Run all** only after reading the setup instructions.
3. Demo 2 and Demo 3 require your own Groq API key.
4. Never paste an API key into a code cell, notebook text, screenshot, or public repository.
5. In Colab, use **Secrets** (key icon in the left sidebar) and add a secret named `GROQ_API_KEY`. Grant this notebook access when prompted.

**Important Colab distinction:** Colab provides a managed Python runtime. It is excellent for executing today's code, but it does not teach local Python version selection or the normal local `venv` activation workflow. We will explain those concepts today and practise them in VS Code in the next class.

# DEMO 1 — Setting Up Your Python AI Project from Scratch

### Objective
By the end of this demonstration, you will be able to:
- Create a clean Python AI project folder from scratch.
- Understand what a virtual environment does and why isolated development matters.
- Create a professional project structure for AI applications.
- Install basic packages required for AI and API-based projects.
- Create and run a simple Python starter script.
- Prepare the project for future API key management, notebooks, GitHub usage and LLM integration.

### What you will learn
You will learn the purpose of `src/`, `notebooks/`, `data/`, `tests/`, `requirements.txt`, `.gitignore` and `.env.example`.

### Instructor script — say before starting
“Before we build an AI agent, we need a place to build it. Imagine a team where everyone saves code, datasets and configuration files wherever they like. Very quickly, nobody knows which file is the latest or how to run the project. Professional engineering starts with structure. Today, in Colab, we will create that structure and run a small Python program. Colab manages the runtime for us; in a later class, we will create and activate a local virtual environment in VS Code.”

### Step 1 — Inspect the Colab Python runtime
Colab already provides Python and pip. These commands let us inspect the runtime. This is not the same as installing or switching Python versions with `pyenv` on your own laptop.

In [ ]:
import sys
import platform
print("Python:", sys.version)
print("Platform:", platform.platform())
!python --version
!python -m pip --version

### Step 2 — Create a project workspace
We will create the same logical project layout used in a local Python project. Colab's working directory is temporary and may be reset when the runtime is recycled.

In [ ]:
from pathlib import Path

PROJECT = Path("/content/python-ai-project")
folders = ["src", "notebooks", "data", "tests"]

PROJECT.mkdir(parents=True, exist_ok=True)
for folder in folders:
    (PROJECT / folder).mkdir(exist_ok=True)

print("Project created at:", PROJECT)
for path in sorted(PROJECT.rglob("*")):
    print(path.relative_to(PROJECT))

### Step 3 — Create the starter files
The `.env.example` file is a template only. Do not put a real API key in it.

In [ ]:
starter_files = {
    "src/main.py": '''def main():
    print("Python AI project setup is complete.")
    print("The environment is ready for AI development.")

if __name__ == "__main__":
    main()
''',
    "README.md": "# Python AI Project\n\nA starter project for learning Python-based AI application development.\n",
    "requirements.txt": "python-dotenv\nrequests\ngroq\njupyterlab\nipykernel\n",
    ".gitignore": ".env\n.venv/\n__pycache__/\n*.py[cod]\n.ipynb_checkpoints/\n",
    ".env.example": "GROQ_API_KEY=replace_with_your_key\nMODEL_NAME=openai/gpt-oss-120b\nENVIRONMENT=development\n"
}

for relative_path, content in starter_files.items():
    target = PROJECT / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")

print("Starter files created.")

### Step 4 — Understand the project structure
- `src/`: application source code.
- `notebooks/`: experiments and interactive analysis.
- `data/`: development data files.
- `tests/`: tests that validate code.
- `requirements.txt`: packages needed to recreate the project environment.
- `.gitignore`: files Git should not track.
- `.env.example`: names of required configuration variables, without real secrets.
- `.venv/`: a local virtual environment; do not create or rely on it inside this hosted Colab runtime.

In [ ]:
print((PROJECT / "src/main.py").read_text())

### Step 5 — Install the packages used in this module
In Colab, install packages into the current runtime. In a local VS Code project, you would first activate `.venv`, then install from `requirements.txt`.

In [ ]:
%pip install -q groq python-dotenv requests jupyterlab ipykernel
print("Required packages installed in this Colab runtime.")

### Step 6 — Run the starter script

In [ ]:
!python /content/python-ai-project/src/main.py

### Step 7 — Check the dependency file and key template

In [ ]:
print("requirements.txt:")
print((PROJECT / "requirements.txt").read_text())

print(".env.example (template only):")
print((PROJECT / ".env.example").read_text())

### Demo 1 — Check your understanding
1. Why do we keep each project in its own folder?
2. What problem does a virtual environment solve?
3. What belongs in `requirements.txt`?
4. Should a real API key ever be placed in `.env.example`?

**Key takeaway:** A well-structured project is easier to understand, reproduce, extend and maintain.

**Bridge to Demo 2:** “We have prepared the workspace. Next, our Python application needs to authenticate with an external AI service. How do we do that without exposing our credentials?”

# DEMO 2 — Managing API Keys Securely with `python-dotenv`

### Objective
By the end of this demonstration, you will be able to:
- Explain why API keys should never be hardcoded in source code.
- Understand environment variables and `.env` files.
- Load environment variables using `python-dotenv`.
- Access a Groq API key from Python code.
- Verify that configuration is loaded without printing the secret.
- Explain how `.gitignore` protects credentials from being committed to Git repositories.

### What you will learn
You will learn the difference between hardcoded credentials and external configuration, how to use Colab Secrets for today's hosted notebook, and how the same application can load a local `.env` file in VS Code.

### Instructor script — say before starting
“An API key is like a password that allows your application to use a service. If you hardcode it and push your code to GitHub, you may expose your account to misuse. The engineering principle is simple: separate secrets from source code. In Colab, the safer place for our real key is Colab Secrets. In a local VS Code project, we commonly use a `.env` file and `python-dotenv`, while ensuring `.env` is ignored by Git.”

### Step 1 — Add your Groq key to Colab Secrets
1. Open the **Secrets** panel using the key icon in Colab's left sidebar.
2. Add a secret named exactly `GROQ_API_KEY`.
3. Paste your own key into the secret value and enable notebook access.
4. Do not put the key in a code cell or print it.

Create a key at https://console.groq.com/keys. You may need an account and access subject to the provider's current terms and limits.

In [ ]:
from google.colab import userdata
import os

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "Add GROQ_API_KEY in Colab Secrets (key icon) and grant this notebook access."
    ) from exc

if not GROQ_API_KEY:
    raise RuntimeError("GROQ_API_KEY is empty. Check your Colab Secret.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["MODEL_NAME"] = "openai/gpt-oss-120b"

print("API key loaded:", bool(os.environ.get("GROQ_API_KEY")))
print("Model configured:", os.environ["MODEL_NAME"])
print("Secret value is intentionally not displayed.")

### Step 2 — Understand the local `.env` pattern
In VS Code, students will create a local `.env` file containing their own key. It should never be committed. For this Colab lesson, we will create a temporary `.env` file from the Colab Secret so we can demonstrate `python-dotenv` without typing the secret into a cell.

In [ ]:
from pathlib import Path

env_file = PROJECT / ".env"
env_file.write_text(
    f"GROQ_API_KEY={GROQ_API_KEY}\n"
    "MODEL_NAME=openai/gpt-oss-120b\n"
    "ENVIRONMENT=development\n",
    encoding="utf-8"
)
print("Temporary .env file created in the Colab runtime.")
print("We will not display its contents.")

### Step 3 — Load variables using python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=env_file, override=True)

api_key = os.getenv("GROQ_API_KEY")
model_name = os.getenv("MODEL_NAME")
environment = os.getenv("ENVIRONMENT")

print("Environment:", environment)
print("Model:", model_name)
print("API key loaded successfully:", bool(api_key))

### Step 4 — Create a reusable configuration module

In [ ]:
config_code = '''from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv(dotenv_path=Path(__file__).resolve().parents[1] / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-oss-120b")
ENVIRONMENT = os.getenv("ENVIRONMENT", "development")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing")
'''

(PROJECT / "src/config.py").write_text(config_code, encoding="utf-8")
print("Created src/config.py")

### Step 5 — Protect secrets with `.gitignore`
`.gitignore` tells Git which untracked files to ignore. It does not remove a secret that was already committed. If a key is exposed, revoke/rotate it immediately.

In [ ]:
gitignore_text = (PROJECT / ".gitignore").read_text(encoding="utf-8")
if ".env" not in gitignore_text.splitlines():
    gitignore_text += "\n.env\n"
(PROJECT / ".gitignore").write_text(gitignore_text, encoding="utf-8")

print((PROJECT / ".gitignore").read_text())

### Step 6 — Make a real LLM request (optional live check)
This verifies that the credential can authenticate to Groq. It may consume account quota and is subject to current model availability and rate limits.

In [ ]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])
response = client.chat.completions.create(
    model=os.environ["MODEL_NAME"],
    messages=[
        {"role": "user", "content": "Explain an AI agent in one clear sentence."}
    ],
    temperature=0.2,
    max_completion_tokens=120
)
print(response.choices[0].message.content)

### Demo 2 — Check your understanding
1. Why is hardcoding an API key unsafe?
2. What does `load_dotenv()` do?
3. Why do we use `os.getenv()`?
4. What is the difference between `.env` and `.env.example`?
5. Does `.gitignore` protect a key that was already committed?

**Key takeaway:** Keep credentials outside source code, do not print or share them, and protect local secret files from version control.

**Bridge to Demo 3:** “Our application can now securely talk to an LLM. But what if we need five independent answers? Do we wait for each one, or can we coordinate the requests more efficiently?”

# DEMO 3 — Running Concurrent LLM Calls with `asyncio`

### Objective
By the end of this demonstration, you will be able to:
- Explain why concurrent execution is useful for multiple LLM requests.
- Use Python `asyncio` to run multiple Groq LLM calls concurrently.
- Compare sequential and concurrent API execution time.
- Build reusable asynchronous functions for LLM calls.
- Handle errors safely when multiple async tasks run together.
- Measure performance using Python timing utilities.

### What you will learn
You will learn `async def`, `await`, `asyncio.gather()`, `AsyncGroq`, `time.perf_counter()` and `return_exceptions=True`.

### Instructor script — say before starting
“Imagine an AI research assistant that must summarize five documents. If it waits for document one before starting document two, total waiting time can add up. Most of the time in an API call is spent waiting for a response over the network. Asynchronous programming lets independent requests make progress during overlapping periods. We will run the same five prompts sequentially and concurrently, then compare actual elapsed time. The result will vary with network conditions, model load and rate limits.”

### Step 1 — Configure the async Groq client

In [ ]:
import asyncio
import time
from groq import AsyncGroq

API_KEY = os.environ.get("GROQ_API_KEY")
MODEL_NAME = os.environ.get("MODEL_NAME", "openai/gpt-oss-120b")

if not API_KEY:
    raise RuntimeError("GROQ_API_KEY is missing. Complete Demo 2 first.")

async_client = AsyncGroq(api_key=API_KEY)

prompts = [
    "Explain AI agents in one sentence.",
    "Explain prompt engineering in one sentence.",
    "Explain retrieval-augmented generation (RAG) in one sentence.",
    "Explain tool calling in one sentence.",
    "Explain agent memory in one sentence."
]

print(f"Prepared {len(prompts)} independent prompts.")

### Step 2 — Create one reusable asynchronous LLM function

In [ ]:
async def call_llm(prompt):
    response = await async_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_completion_tokens=120
    )
    return response.choices[0].message.content

### Step 3 — Run calls sequentially
Each request is awaited before the next request begins. This is easy to understand, but network waiting time accumulates.

In [ ]:
async def run_sequential():
    start = time.perf_counter()
    results = []

    for prompt in prompts:
        results.append(await call_llm(prompt))

    elapsed = time.perf_counter() - start
    return results, elapsed

sequential_results, sequential_time = await run_sequential()

for index, result in enumerate(sequential_results, start=1):
    print(f"\nResponse {index}:\n{result}")

print(f"\nSequential time: {sequential_time:.2f} seconds")

### Step 4 — Run calls concurrently with `asyncio.gather()`
We create the coroutine tasks first, then gather them. `gather()` returns results in the same order as the input awaitables.

In [ ]:
async def run_concurrent():
    start = time.perf_counter()

    tasks = [call_llm(prompt) for prompt in prompts]
    results = await asyncio.gather(*tasks)

    elapsed = time.perf_counter() - start
    return results, elapsed

concurrent_results, concurrent_time = await run_concurrent()

for index, result in enumerate(concurrent_results, start=1):
    print(f"\nResponse {index}:\n{result}")

print(f"\nConcurrent time: {concurrent_time:.2f} seconds")

### Step 5 — Compare the timings
Do not promise a fixed speed-up. The measured result depends on API latency, network conditions, model load, rate limits and service-side scheduling.

In [ ]:
print(f"Sequential: {sequential_time:.2f} seconds")
print(f"Concurrent: {concurrent_time:.2f} seconds")

if concurrent_time > 0:
    print(f"Observed speed-up: {sequential_time / concurrent_time:.2f}x")
else:
    print("Concurrent duration was too small to calculate a ratio.")

### Step 6 — Handle errors safely
With `return_exceptions=True`, one failed request is returned as an exception result instead of immediately cancelling the whole gather operation. Production systems should also consider timeouts, bounded concurrency, retries with backoff and provider rate limits.

In [ ]:
async def call_llm_safely(prompt):
    try:
        return await asyncio.wait_for(call_llm(prompt), timeout=45)
    except Exception as exc:
        return exc

async def run_concurrent_with_errors():
    tasks = [call_llm_safely(prompt) for prompt in prompts]
    return await asyncio.gather(*tasks, return_exceptions=True)

safe_results = await run_concurrent_with_errors()

for index, result in enumerate(safe_results, start=1):
    print(f"\nTask {index}:")
    if isinstance(result, Exception):
        print("Request failed:", type(result).__name__, str(result)[:250])
    else:
        print(result)

### Demo 3 — Check your understanding
1. What is the difference between sequential and concurrent execution?
2. What does `async def` define?
3. What does `await` do?
4. What does `asyncio.gather()` help us do?
5. Why can concurrent I/O-bound LLM calls reduce elapsed time?
6. Why do production applications need timeouts and concurrency limits?

**Key takeaway:** For independent network-bound LLM requests, concurrency can reduce total waiting time. It does not guarantee faster execution for every workload.

---

# Final recap — What you have built

1. **Demo 1:** A structured Python AI project workspace.
2. **Demo 2:** A safer configuration pattern for connecting to an LLM.
3. **Demo 3:** Sequential and concurrent LLM execution, with timing and basic error handling.

## Before the next class
Install locally:
- Visual Studio Code: https://code.visualstudio.com/
- Python 3.11 or 3.12: https://www.python.org/downloads/
- Git: https://git-scm.com/downloads/
- Create a GitHub account: https://github.com/

**Today's goal was to understand the concepts. Next class, we will engineer them professionally in a local VS Code environment.**